# 02 - Kaggle Supervised Baseline Runner

This notebook runs the four supervised baseline experiments on Kaggle GPU:

- `resnet18_none`
- `resnet18_imagenet`
- `vit_s16_none`
- `vit_s16_imagenet`

The notebook only prepares the Kaggle environment and calls repository scripts. It does not contain model training logic.

## 1. Enable Kaggle GPU

Before running, open **Settings** in Kaggle and select a GPU accelerator. T4 is enough for these baseline runs.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import pandas as pd

print('Python:', sys.version)
print('Working directory:', Path.cwd())

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Working directory: /kaggle/working


## 2. Clone Or Pull Repository

This keeps the Kaggle working directory synchronized with the GitHub repository.

In [ ]:
REPO_URL = 'https://github.com/tlinhevg05/contrastive-synthesis-medcls_CVProject.git'
REPO_ROOT = Path('/kaggle/working/contrastive-synthesis-medcls_CVProject')

if REPO_ROOT.exists():
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_ROOT)], check=True)

os.chdir(REPO_ROOT)
print('REPO_ROOT:', REPO_ROOT)

Cloning into '/kaggle/working/contrastive-synthesis-medcls_CVProject'...


REPO_ROOT: /kaggle/working/contrastive-synthesis-medcls_CVProject


Updating files: 100% (21260/21260), done.


## 3. Install Minimal Dependencies

Kaggle already provides PyTorch. This cell installs only the packages commonly needed by the training/evaluation scripts.

In [ ]:
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'timm', 'scikit-learn', 'matplotlib', 'pandas', 'Pillow', 'pyyaml'
], check=True)

import torch
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 94.9 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav

Torch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


## 4. Editable Runner Variables

Set the Kaggle dataset path and choose which baseline experiments to run. Add your Kaggle dataset named `medcls-cvproject` from the right panel before running data checks.

In [ ]:
# Kaggle dataset should contain:
# /kaggle/input/medcls-cvproject/data/processed/labelled_4232
# /kaggle/input/medcls-cvproject/data/manifests/train.csv, val.csv, test.csv
DATA_ROOT = Path('/kaggle/input/datasets/tlinhevg05/medcls-cvproject/data')
LABELLED_SOURCE = DATA_ROOT / 'processed/labelled_4232'
MANIFEST_SOURCE = DATA_ROOT / 'manifests'

OUTPUT_ROOT = Path('/kaggle/working/results/experiments')

RUN_RESNET_NONE = True
RUN_RESNET_IMAGENET = True
RUN_VIT_NONE = True
RUN_VIT_IMAGENET = True

# Use 1 for smoke test. Use None for config/default epochs.
EPOCH_OVERRIDE = None
NUM_WORKERS = 2

print('DATA_ROOT:', DATA_ROOT)
print('LABELLED_SOURCE exists:', LABELLED_SOURCE.exists())
print('MANIFEST_SOURCE exists:', MANIFEST_SOURCE.exists())
print('OUTPUT_ROOT:', OUTPUT_ROOT)

DATA_ROOT: /kaggle/input/datasets/tlinhevg05/medcls-cvproject/data
LABELLED_SOURCE exists: True
MANIFEST_SOURCE exists: True
OUTPUT_ROOT: /kaggle/working/results/experiments


## 5. Link Data And Manifests

The repo scripts expect data under `data/processed` and manifests under `data/manifests`. This cell symlinks Kaggle input data into the cloned repository when available.

In [ ]:
def replace_path(target: Path, source: Path):
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists() or target.is_symlink():
        if target.is_symlink() or target.is_file():
            target.unlink()
        else:
            shutil.rmtree(target)
    if source.exists():
        os.symlink(source, target, target_is_directory=source.is_dir())
        print('Linked:', target, '->', source)
    else:
        print('WARNING: source missing:', source)

replace_path(REPO_ROOT / 'data/processed/labelled_4232', LABELLED_SOURCE)

(REPO_ROOT / 'data/manifests').mkdir(parents=True, exist_ok=True)
for name in ['train.csv', 'val.csv', 'test.csv', 'labelled_all.csv', 'split_summary.json']:
    src = MANIFEST_SOURCE / name
    dst = REPO_ROOT / 'data/manifests' / name
    if src.exists():
        if dst.exists() or dst.is_symlink():
            dst.unlink()
        os.symlink(src, dst)
        print('Linked manifest:', dst, '->', src)
    elif dst.exists():
        print('Using repo manifest:', dst)
    else:
        print('WARNING: missing manifest:', src)

print('\nRepository data snapshot:')
subprocess.run(['find', 'data', '-maxdepth', '3', '-type', 'd'], check=False)
subprocess.run(['ls', '-lh', 'data/manifests'], check=False)

Linked: /kaggle/working/contrastive-synthesis-medcls_CVProject/data/processed/labelled_4232 -> /kaggle/input/datasets/tlinhevg05/medcls-cvproject/data/processed/labelled_4232
Linked manifest: /kaggle/working/contrastive-synthesis-medcls_CVProject/data/manifests/train.csv -> /kaggle/input/datasets/tlinhevg05/medcls-cvproject/data/manifests/train.csv
Linked manifest: /kaggle/working/contrastive-synthesis-medcls_CVProject/data/manifests/val.csv -> /kaggle/input/datasets/tlinhevg05/medcls-cvproject/data/manifests/val.csv
Linked manifest: /kaggle/working/contrastive-synthesis-medcls_CVProject/data/manifests/test.csv -> /kaggle/input/datasets/tlinhevg05/medcls-cvproject/data/manifests/test.csv
Linked manifest: /kaggle/working/contrastive-synthesis-medcls_CVProject/data/manifests/labelled_all.csv -> /kaggle/input/datasets/tlinhevg05/medcls-cvproject/data/manifests/labelled_all.csv
Linked manifest: /kaggle/working/contrastive-synthesis-medcls_CVProject/data/manifests/split_summary.json -> /kag

CompletedProcess(args=['ls', '-lh', 'data/manifests'], returncode=0)

## 6. Lightweight Checks

These checks should pass before launching training.

In [ ]:
from pathlib import Path
import subprocess

print('Kaggle input root:')
subprocess.run(['find', '/kaggle/input', '-maxdepth', '4', '-type', 'd'], check=False)

print('\nSearch labelled_4232:')
matches = list(Path('/kaggle/input').rglob('labelled_4232'))
print(matches)

print('\nSearch manifests:')
matches = list(Path('/kaggle/input').rglob('manifests'))
print(matches)

print('\nSearch csv files:')
csvs = list(Path('/kaggle/input').rglob('*.csv'))
for p in csvs[:30]:
    print(p)

Kaggle input root:

Search labelled_4232:
/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/tlinhevg05
/kaggle/input/datasets/tlinhevg05/medcls-cvproject
/kaggle/input/datasets/tlinhevg05/medcls-cvproject/data
[PosixPath('/kaggle/input/datasets/tlinhevg05/medcls-cvproject/data/processed/labelled_4232')]

Search manifests:
[PosixPath('/kaggle/input/datasets/tlinhevg05/medcls-cvproject/data/manifests')]

Search csv files:
/kaggle/input/datasets/tlinhevg05/medcls-cvproject/data/manifests/synthetic_dcgan.csv
/kaggle/input/datasets/tlinhevg05/medcls-cvproject/data/manifests/val.csv
/kaggle/input/datasets/tlinhevg05/medcls-cvproject/data/manifests/labelled_all.csv
/kaggle/input/datasets/tlinhevg05/medcls-cvproject/data/manifests/train.csv
/kaggle/input/datasets/tlinhevg05/medcls-cvproject/data/manifests/test.csv
/kaggle/input/datasets/tlinhevg05/medcls-cvproject/data/processed/labelled_4232_manifest.csv
/kaggle/input/datasets/tlinhevg05/medcls-cvproject/data/processed/unlabelled_16

In [ ]:
subprocess.run([sys.executable, 'scripts/check_experiment_inputs.py'], check=True)
subprocess.run([
    sys.executable, '-m', 'py_compile',
    'scripts/run_classification_resnet.py',
    'scripts/evaluate_classification_resnet.py',
    'scripts/run_classification_vit.py',
    'scripts/evaluate_classification_vit.py',
], check=True)


Experiment Input Check Report
[PASS] common config
  - loaded configs/experiments/common.yaml
[PASS] fixed supervised manifests
  - train: {'COVID': 578, 'Lung_Opacity': 961, 'Viral_Pneumonia': 215, 'Normal': 1630} total=3384
  - val: {'COVID': 72, 'Lung_Opacity': 120, 'Viral_Pneumonia': 26, 'Normal': 203} total=421
  - test: {'COVID': 73, 'Lung_Opacity': 121, 'Viral_Pneumonia': 28, 'Normal': 205} total=427
[PASS] experiment config files
[PASS] resnet18_covidqu
  - planned output_dir: results/experiments/resnet18_covidqu
[PASS] resnet18_covidqu_syn
  - synthetic_dcgan: {'COVID': 1000, 'Lung_Opacity': 1000, 'Viral_Pneumonia': 1000, 'Normal': 1000} total=4000
  - planned output_dir: results/experiments/resnet18_covidqu_syn
[PASS] resnet18_imagenet
  - no contrastive pretraining data required
  - planned output_dir: results/experiments/resnet18_imagenet
[PASS] resnet18_imagenet_covidqu
  - planned output_dir: results/experiments/resnet18_imagenet_covidqu
[PASS] resnet18_imagenet_covidqu_

CompletedProcess(args=['/usr/bin/python3', '-m', 'py_compile', 'scripts/run_classification_resnet.py', 'scripts/evaluate_classification_resnet.py', 'scripts/run_classification_vit.py', 'scripts/evaluate_classification_vit.py'], returncode=0)

## 7. Helper Function

In [ ]:
def run_cmd(cmd):
    print('Running:')
    print(' '.join(map(str, cmd)))
    subprocess.run(list(map(str, cmd)), check=True)

def epoch_args():
    return [] if EPOCH_OVERRIDE is None else ['--epochs', str(EPOCH_OVERRIDE)]

def show_metrics(experiment_id):
    metrics_path = OUTPUT_ROOT / experiment_id / 'metrics.json'
    if not metrics_path.exists():
        print('Missing metrics:', metrics_path)
        return None
    metrics = json.loads(metrics_path.read_text())
    row = {'experiment_id': experiment_id, **metrics}
    display(pd.DataFrame([row]))
    return row

## 8. Experiment: resnet18_none

Train ResNet18 from random initialization on the fixed real labeled train/val/test manifests.

In [ ]:
EXP = 'resnet18_none'
if RUN_RESNET_NONE:
    run_cmd([
        sys.executable, 'scripts/run_classification_resnet.py',
        '--config', 'configs/experiments/resnet18/none.yaml',
        '--manifest-dir', 'data/manifests',
        '--output-dir', OUTPUT_ROOT / EXP,
        '--num-workers', NUM_WORKERS,
        *epoch_args(),
    ])
    show_metrics(EXP)
else:
    print('Skipping', EXP)

Running:
/usr/bin/python3 scripts/run_classification_resnet.py --config configs/experiments/resnet18/none.yaml --manifest-dir data/manifests --output-dir /kaggle/working/results/experiments/resnet18_none --num-workers 2
Epoch 1/70 train_loss=0.7794 val_loss=0.9030 val_acc=0.6437 val_f1_macro=0.6379
Epoch 2/70 train_loss=0.5383 val_loss=0.7481 val_acc=0.6960 val_f1_macro=0.7158
Epoch 3/70 train_loss=0.4357 val_loss=0.5735 val_acc=0.8052 val_f1_macro=0.8079
Epoch 4/70 train_loss=0.3596 val_loss=0.7348 val_acc=0.7672 val_f1_macro=0.7471
Epoch 5/70 train_loss=0.2996 val_loss=0.5304 val_acc=0.8147 val_f1_macro=0.8129
Epoch 6/70 train_loss=0.2250 val_loss=0.5552 val_acc=0.8076 val_f1_macro=0.8212
Epoch 7/70 train_loss=0.2013 val_loss=0.8958 val_acc=0.7886 val_f1_macro=0.7635
Epoch 8/70 train_loss=0.1762 val_loss=0.5889 val_acc=0.7815 val_f1_macro=0.7856
Epoch 9/70 train_loss=0.1296 val_loss=0.8919 val_acc=0.7078 val_f1_macro=0.7335
Epoch 10/70 train_loss=0.0998 val_loss=0.6055 val_acc=0.8361

,experiment_id,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted,best_epoch,best_val_f1_macro
0,resnet18_none,0.882904,0.909286,0.890551,0.898092,0.88438,0.882904,0.881625,49,0.874358


## 9. Experiment: resnet18_imagenet

Fine-tune ImageNet-pretrained ResNet18 on the fixed real labeled train/val/test manifests.

In [ ]:
EXP = 'resnet18_imagenet'
if RUN_RESNET_IMAGENET:
    run_cmd([
        sys.executable, 'scripts/run_classification_resnet.py',
        '--config', 'configs/experiments/resnet18/imagenet.yaml',
        '--manifest-dir', 'data/manifests',
        '--output-dir', OUTPUT_ROOT / EXP,
        '--num-workers', NUM_WORKERS,
        *epoch_args(),
    ])
    show_metrics(EXP)
else:
    print('Skipping', EXP)

Running:
/usr/bin/python3 scripts/run_classification_resnet.py --config configs/experiments/resnet18/imagenet.yaml --manifest-dir data/manifests --output-dir /kaggle/working/results/experiments/resnet18_imagenet --num-workers 2
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 143MB/s]


Epoch 1/70 train_loss=0.7623 val_loss=0.5737 val_acc=0.8005 val_f1_macro=0.7927
Epoch 2/70 train_loss=0.4226 val_loss=0.4227 val_acc=0.8337 val_f1_macro=0.8294
Epoch 3/70 train_loss=0.3018 val_loss=0.3598 val_acc=0.8670 val_f1_macro=0.8739
Epoch 4/70 train_loss=0.2355 val_loss=0.3260 val_acc=0.8789 val_f1_macro=0.8840
Epoch 5/70 train_loss=0.1917 val_loss=0.3105 val_acc=0.8860 val_f1_macro=0.8892
Epoch 6/70 train_loss=0.1554 val_loss=0.2959 val_acc=0.8979 val_f1_macro=0.9011
Epoch 7/70 train_loss=0.1309 val_loss=0.2975 val_acc=0.9026 val_f1_macro=0.9073
Epoch 8/70 train_loss=0.1173 val_loss=0.2827 val_acc=0.9074 val_f1_macro=0.9126
Epoch 9/70 train_loss=0.0888 val_loss=0.2883 val_acc=0.9050 val_f1_macro=0.9124
Epoch 10/70 train_loss=0.0721 val_loss=0.2857 val_acc=0.9097 val_f1_macro=0.9214
Epoch 11/70 train_loss=0.0604 val_loss=0.2842 val_acc=0.9121 val_f1_macro=0.9234
Epoch 12/70 train_loss=0.0471 val_loss=0.2867 val_acc=0.9050 val_f1_macro=0.9145
Epoch 13/70 train_loss=0.0416 val_los

,experiment_id,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted,best_epoch,best_val_f1_macro
0,resnet18_imagenet,0.93911,0.957435,0.950671,0.953937,0.939285,0.93911,0.93906,36,0.932232


## 10. Experiment: vit_s16_none

Train ViT-S/16 from random initialization on the fixed real labeled train/val/test manifests.

In [ ]:
EXP = 'vit_s16_none'
if RUN_VIT_NONE:
    run_cmd([
        sys.executable, 'scripts/run_classification_vit.py',
        '--config', 'configs/experiments/vit_s16/none.yaml',
        '--manifest-dir', 'data/manifests',
        '--output-dir', OUTPUT_ROOT / EXP,
        '--num-workers', NUM_WORKERS,
        *epoch_args(),
    ])
    show_metrics(EXP)
else:
    print('Skipping', EXP)

Running:
/usr/bin/python3 scripts/run_classification_vit.py --config configs/experiments/vit_s16/none.yaml --manifest-dir data/manifests --output-dir /kaggle/working/results/experiments/vit_s16_none --num-workers 2
Epoch 1/50 train_loss=1.1640 val_loss=1.1104 val_acc=0.5392 val_f1_macro=0.3963
Epoch 2/50 train_loss=1.0515 val_loss=0.9895 val_acc=0.5748 val_f1_macro=0.3213
Epoch 3/50 train_loss=0.9752 val_loss=0.8833 val_acc=0.6413 val_f1_macro=0.5392
Epoch 4/50 train_loss=0.8969 val_loss=0.9094 val_acc=0.6128 val_f1_macro=0.5896
Epoch 5/50 train_loss=0.8538 val_loss=0.8358 val_acc=0.6651 val_f1_macro=0.5747
Epoch 6/50 train_loss=0.8078 val_loss=0.8056 val_acc=0.6841 val_f1_macro=0.6104
Epoch 7/50 train_loss=0.7618 val_loss=0.7978 val_acc=0.6722 val_f1_macro=0.5639
Epoch 8/50 train_loss=0.7164 val_loss=0.7232 val_acc=0.7150 val_f1_macro=0.6597
Epoch 9/50 train_loss=0.6845 val_loss=0.6348 val_acc=0.7672 val_f1_macro=0.7768
Epoch 10/50 train_loss=0.6373 val_loss=0.6478 val_acc=0.7458 val_

,experiment_id,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted,best_epoch,best_val_f1_macro
0,vit_s16_none,0.777518,0.827831,0.762732,0.786988,0.785965,0.777518,0.772104,40,0.820963


## 11. Experiment: vit_s16_imagenet

Fine-tune ImageNet-pretrained ViT-S/16 on the fixed real labeled train/val/test manifests.

In [ ]:
EXP = 'vit_s16_imagenet'
if RUN_VIT_IMAGENET:
    run_cmd([
        sys.executable, 'scripts/run_classification_vit.py',
        '--config', 'configs/experiments/vit_s16/imagenet.yaml',
        '--manifest-dir', 'data/manifests',
        '--output-dir', OUTPUT_ROOT / EXP,
        '--num-workers', NUM_WORKERS,
        *epoch_args(),
    ])
    show_metrics(EXP)
else:
    print('Skipping', EXP)

Running:
/usr/bin/python3 scripts/run_classification_vit.py --config configs/experiments/vit_s16/imagenet.yaml --manifest-dir data/manifests --output-dir /kaggle/working/results/experiments/vit_s16_imagenet --num-workers 2
Epoch 1/50 train_loss=0.7012 val_loss=0.4470 val_acc=0.8385 val_f1_macro=0.8473
Epoch 2/50 train_loss=0.3035 val_loss=0.3675 val_acc=0.8646 val_f1_macro=0.8622
Epoch 3/50 train_loss=0.1755 val_loss=0.3313 val_acc=0.8765 val_f1_macro=0.8772
Epoch 4/50 train_loss=0.1063 val_loss=0.3683 val_acc=0.8717 val_f1_macro=0.8851
Epoch 5/50 train_loss=0.0892 val_loss=0.3016 val_acc=0.9026 val_f1_macro=0.9060
Epoch 6/50 train_loss=0.0420 val_loss=0.3555 val_acc=0.8907 val_f1_macro=0.8988
Epoch 7/50 train_loss=0.0276 val_loss=0.3329 val_acc=0.9026 val_f1_macro=0.9101
Epoch 8/50 train_loss=0.0241 val_loss=0.3476 val_acc=0.9121 val_f1_macro=0.9184
Epoch 9/50 train_loss=0.0303 val_loss=0.3689 val_acc=0.9002 val_f1_macro=0.9123
Epoch 10/50 train_loss=0.0108 val_loss=0.3767 val_acc=0.8

,experiment_id,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted,best_epoch,best_val_f1_macro
0,vit_s16_imagenet,0.906323,0.906358,0.915218,0.908575,0.908495,0.906323,0.905627,8,0.918406


## 12. Summary Table

In [ ]:
rows = []
for exp in ['resnet18_none', 'resnet18_imagenet', 'vit_s16_none', 'vit_s16_imagenet']:
    metrics_path = OUTPUT_ROOT / exp / 'metrics.json'
    if metrics_path.exists():
        metrics = json.loads(metrics_path.read_text())
        rows.append({'experiment_id': exp, **metrics})

summary = pd.DataFrame(rows)
if len(summary):
    display(summary)
    summary_path = OUTPUT_ROOT / 'supervised_baseline_summary.csv'
    summary.to_csv(summary_path, index=False)
    print('Saved:', summary_path)
else:
    print('No completed baseline metrics found yet.')

,experiment_id,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted,best_epoch,best_val_f1_macro
0,resnet18_none,0.882904,0.909286,0.890551,0.898092,0.884380,0.882904,0.881625,49,0.874358
1,resnet18_imagenet,0.939110,0.957435,0.950671,0.953937,0.939285,0.939110,0.939060,36,0.932232
2,vit_s16_none,0.777518,0.827831,0.762732,0.786988,0.785965,0.777518,0.772104,40,0.820963
3,vit_s16_imagenet,0.906323,0.906358,0.915218,0.908575,0.908495,0.906323,0.905627,8,0.918406


Saved: /kaggle/working/results/experiments/supervised_baseline_summary.csv


## 13. Package Results

Kaggle output files can be downloaded from `/kaggle/working`. This cell creates one zip archive for the baseline outputs.

In [ ]:
zip_base = Path('/kaggle/working/supervised_baseline_results')
if zip_base.with_suffix('.zip').exists():
    zip_base.with_suffix('.zip').unlink()
shutil.make_archive(str(zip_base), 'zip', root_dir=OUTPUT_ROOT.parent, base_dir='experiments')
print('Created:', zip_base.with_suffix('.zip'))

Created: /kaggle/working/supervised_baseline_results.zip
